# RAG Pipeline vs Radiologist Description Comparison (GPT-4 as Judge)

This notebook compares generated chest X-ray descriptions from `rag_pipeline` with professional radiologist reference descriptions using **GPT-4 as an AI judge**.

- Reference descriptions (`fully correct`): `data/test_metadata.json`
- Generated descriptions: `data/test_outputs.json`

The notebook aligns cases by filename, asks GPT-4 to score each generated report against the radiologist reference, and summarizes best/worst matching examples.

Before running the judging cells, set your API key in the environment:

`OPENAI_API_KEY=...`

In [5]:
from pathlib import Path
import json
import os
import re
import time

import pandas as pd
from openai import OpenAI

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
metadata_path = ROOT / "data" / "test_metadata.json"
outputs_path = ROOT / "data" / "test_outputs.json"

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

with open(outputs_path, "r", encoding="utf-8") as f:
    outputs = json.load(f)

print(f"Loaded reference records: {len(metadata)}")
print(f"Loaded generated records: {len(outputs)}")

if not os.getenv("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY is not set. GPT-4 judging cells will fail until it is configured.")

Loaded reference records: 202
Loaded generated records: 202


In [6]:
DISCLAIMER = "For clinician review only; not a final medical report."

ref_df = pd.DataFrame(metadata)
ref_df["filename"] = ref_df["image_path"].str.split("/").str[-1]
ref_df = ref_df[["filename", "image_path", "short_description", "full_description"]]
ref_df = ref_df.rename(columns={"full_description": "reference_description"})

gen_df = pd.DataFrame(outputs)
gen_df["filename"] = gen_df["filename"].fillna(gen_df["relative_path"].str.split("/").str[-1])

def clean_generated_text(text: str) -> str:
    if pd.isna(text):
        return ""
    out = str(text).replace(DISCLAIMER, "")
    out = re.sub(r"\s+", " ", out).strip()
    return out

gen_df["generated_description_raw"] = gen_df["generated_description"].fillna("")
gen_df["generated_description"] = gen_df["generated_description_raw"].map(clean_generated_text)
gen_df["contains_disclaimer"] = gen_df["generated_description_raw"].str.contains(
    re.escape(DISCLAIMER), regex=True, na=False
)

keep_cols = [
    "filename",
    "relative_path",
    "generated_description_raw",
    "generated_description",
    "quality_score",
    "quality_approved",
    "contains_disclaimer",
]
gen_df = gen_df[keep_cols]

In [7]:
comparison_df = ref_df.merge(gen_df, on="filename", how="left", indicator=True)

print("Merge status counts:")
print(comparison_df["_merge"].value_counts(dropna=False))
print()
print(f"Matched cases: {(comparison_df['_merge'] == 'both').sum()}")
print(f"Reference-only cases: {(comparison_df['_merge'] == 'left_only').sum()}")

matched_df = comparison_df[comparison_df["_merge"] == "both"].copy()
matched_df.head(3)

Merge status counts:
_merge
both          202
left_only       0
right_only      0
Name: count, dtype: int64

Matched cases: 202
Reference-only cases: 0


,filename,image_path,short_description,reference_description,relative_path,generated_description_raw,generated_description,quality_score,quality_approved,contains_disclaimer,_merge
0,PATIENT_0657_1.dcm,test/PATIENT_0657_1.dcm,W Klp. na stojaco pa,"Chest X-ray, PA and lateral projection. Under ...",PATIENT_0657_1.dcm,"Chest X-ray, PA projection. Lung fields withou...","Chest X-ray, PA projection. Lung fields withou...",0.9,True,True,both
1,PATIENT_0563_1.dcm,test/PATIENT_0563_1.dcm,W Klp. na stojaco pa,"Chest X-ray, PA projection. Lung fields withou...",PATIENT_0563_1.dcm,"1. Chest X-ray, PA projection.\n2. Lung fields...","1. Chest X-ray, PA projection. 2. Lung fields ...",0.9,True,True,both
2,PATIENT_0722_1.dcm,test/PATIENT_0722_1.dcm,W Klp. na stojaco pa,"1. Chest X-ray, PA and lateral projection. 2. ...",PATIENT_0722_1.dcm,"1. Chest X-ray, PA projection. \n2. Lung field...","1. Chest X-ray, PA projection. 2. Lung fields ...",0.9,True,True,both


In [14]:
JUDGE_MODEL = "gpt-4.1"
MAX_CASES = None  # Set e.g. 30 for a cheaper, faster pilot run.
SLEEP_BETWEEN_REQUESTS_SEC = 0.0

client = OpenAI()

SYSTEM_PROMPT = (
    "You are an expert radiologist evaluating chest X-ray reports for clinical accuracy and completeness. "
    "Follow the requested output format exactly."
)

PROMPT_TEMPLATE = """
You are an expert radiologist evaluating chest X-ray reports for clinical accuracy and completeness. Your task is to compare an AI-generated report against a reference report written by a licensed radiologist and assign a similarity score from 0 to 1 (where 1 = identical clinical content and meaning, 0 = completely different or missing critical findings).

CRITERIA FOR SCORING (apply all):
- Key clinical findings (abnormalities, devices, critical observations)
- Accurate description of normal vs abnormal structures
- Completeness of standard chest X-ray evaluation (lungs, hila, heart, aorta, diaphragm, costophrenic angles)
- No hallucinated findings in AI report
- No omission of critical findings from reference
- Technical accuracy of anatomical descriptions

INPUT FORMAT:
Reference (Physician): [INSERT PHYSICIAN REPORT HERE]
AI-Generated: [INSERT AI REPORT HERE]

OUTPUT FORMAT (exact):
Score: X.XX/1.0
Strengths: [2-3 bullet points]
Weaknesses: [2-3 bullet points]
Explanation: [1-2 sentences justifying the score]

EXAMPLE:
Reference: "Chest X-ray, PA projection. Under left diaphragm dome, band of lucency up to 10 mm wide - ? free intraperitoneal gas. Clinical correlation/CT recommended."
AI: "Chest X-ray, PA projection. Lungs clear. Heart size normal."
Score: 0.25/1.0
Strengths: Correctly identifies projection; basic normal findings
Weaknesses: Misses critical free gas finding; no recommendation for further imaging
Explanation: AI completely omitted the most important finding (possible pneumoperitoneum) which requires urgent clinical correlation.

Now score this pair:

Reference (Physician): {reference_report}
AI-Generated: {ai_report}
""".strip()


def _extract_field(text: str, field_name: str) -> str:
    pattern = rf"{field_name}:\s*(.*?)(?=\n[A-Za-z ]+:|\Z)"
    match = re.search(pattern, text, flags=re.DOTALL)
    return match.group(1).strip() if match else ""


def _parse_score(text: str) -> float:
    match = re.search(r"Score:\s*([01](?:\.\d+)?)\s*/\s*1\.0", text)
    if not match:
        raise ValueError(f"Could not parse score from model output:\n{text}")
    return float(match.group(1))


def judge_case_with_gpt4(reference_description: str, generated_description: str) -> dict:
    user_prompt = PROMPT_TEMPLATE.format(
        reference_report=reference_description,
        ai_report=generated_description,
    )

    response = client.responses.create(
        model=JUDGE_MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
    )

    content = response.output_text.strip()
    score = _parse_score(content)
    strengths = _extract_field(content, "Strengths")
    weaknesses = _extract_field(content, "Weaknesses")
    explanation = _extract_field(content, "Explanation")

    return {
        "similarity_score_0_to_1": score,
        "strengths": strengths,
        "weaknesses": weaknesses,
        "explanation": explanation,
        "judge_raw_output": content,
    }

In [15]:
from tqdm import tqdm
rows_to_score = matched_df.copy()
if MAX_CASES is not None:
    rows_to_score = rows_to_score.head(MAX_CASES)

judge_rows = []
for idx, row in tqdm(rows_to_score.reset_index(drop=True).iterrows()):
    result = judge_case_with_gpt4(
        reference_description=row["reference_description"],
        generated_description=row["generated_description"],
    )
    judge_rows.append(
        {
            "filename": row["filename"],
            "similarity_score_0_to_1": result["similarity_score_0_to_1"],
            "strengths": result["strengths"],
            "weaknesses": result["weaknesses"],
            "explanation": result["explanation"],
            "judge_raw_output": result["judge_raw_output"],
        }
    )

    if (idx + 1) % 10 == 0:
        print(f"Scored {idx + 1}/{len(rows_to_score)} cases...")
    if SLEEP_BETWEEN_REQUESTS_SEC > 0:
        time.sleep(SLEEP_BETWEEN_REQUESTS_SEC)

judge_df = pd.DataFrame(judge_rows)
scored_df = rows_to_score.merge(judge_df, on="filename", how="left")

print(f"Scored cases: {len(scored_df)}")
scored_df.head(3)

10it [00:45,  4.20s/it]

Scored 10/202 cases...


20it [01:55,  5.43s/it]

Scored 20/202 cases...


30it [02:41,  4.72s/it]

...

200it [17:58,  7.60s/it]

Scored 200/202 cases...


202it [18:05,  5.37s/it]

Scored cases: 202


,filename,image_path,short_description,reference_description,relative_path,generated_description_raw,generated_description,quality_score,quality_approved,contains_disclaimer,_merge,similarity_score_0_to_1,strengths,weaknesses,explanation,judge_raw_output
0,PATIENT_0657_1.dcm,test/PATIENT_0657_1.dcm,W Klp. na stojaco pa,"Chest X-ray, PA and lateral projection. Under ...",PATIENT_0657_1.dcm,"Chest X-ray, PA projection. Lung fields withou...","Chest X-ray, PA projection. Lung fields withou...",0.9,True,True,both,0.45,- Accurately describes lung fields as without ...,- Omits the critical finding of subdiaphragmat...,The AI report misses the most clinically signi...,Score: 0.45/1.0 \nStrengths: \n- Accurately ...
1,PATIENT_0563_1.dcm,test/PATIENT_0563_1.dcm,W Klp. na stojaco pa,"Chest X-ray, PA projection. Lung fields withou...",PATIENT_0563_1.dcm,"1. Chest X-ray, PA projection.\n2. Lung fields...","1. Chest X-ray, PA projection. 2. Lung fields ...",0.9,True,True,both,0.95,- Accurately covers all key anatomical structu...,"- Omits mention of ""atherosclerotic"" change in...",The AI report closely matches the reference in...,Score: 0.95/1.0 \nStrengths: \n- Accurately ...
2,PATIENT_0722_1.dcm,test/PATIENT_0722_1.dcm,W Klp. na stojaco pa,"1. Chest X-ray, PA and lateral projection. 2. ...",PATIENT_0722_1.dcm,"1. Chest X-ray, PA projection. \n2. Lung field...","1. Chest X-ray, PA projection. 2. Lung fields ...",0.9,True,True,both,0.75,- Accurately reports absence of infiltrative c...,- Omits mention of tracheostomy tube and its p...,The AI report covers most normal findings and ...,Score: 0.75/1.0 \nStrengths:\n- Accurately re...


In [16]:
summary = scored_df[["similarity_score_0_to_1", "quality_score"]].describe().T

print("GPT-4 judge summary:")
display(summary)

view_cols = [
    "filename",
    "similarity_score_0_to_1",
    "quality_score",
    "quality_approved",
    "contains_disclaimer",
]

print("Lowest GPT-4 similarity score examples:")
display(scored_df.sort_values("similarity_score_0_to_1", ascending=True)[view_cols].head(10))

print("Highest GPT-4 similarity score examples:")
display(scored_df.sort_values("similarity_score_0_to_1", ascending=False)[view_cols].head(10))

GPT-4 judge summary:


,count,mean,std,min,25%,50%,75%,max
similarity_score_0_to_1,202.0,0.739604,0.224741,0.0,0.6,0.8,0.95,1.0
quality_score,202.0,0.889604,0.083104,0.2,0.9,0.9,0.90,1.0


Lowest GPT-4 similarity score examples:


,filename,similarity_score_0_to_1,quality_score,quality_approved,contains_disclaimer
96,PATIENT_0397_1.dcm,0.0,0.2,False,False
144,PATIENT_0366_1.dcm,0.1,0.2,False,True
76,PATIENT_0219_1.dcm,0.2,0.9,True,True
54,PATIENT_0661_1.dcm,0.2,0.9,True,True
102,PATIENT_0863_1.dcm,0.2,0.9,True,True
95,PATIENT_0344_1.dcm,0.3,0.9,True,True
114,PATIENT_0430_1.dcm,0.3,0.9,True,True
11,PATIENT_0524_1.dcm,0.3,0.8,True,True
170,PATIENT_0514_1.dcm,0.3,0.9,True,True
193,PATIENT_0018_1.dcm,0.3,0.8,True,True


Highest GPT-4 similarity score examples:


,filename,similarity_score_0_to_1,quality_score,quality_approved,contains_disclaimer
62,PATIENT_0897_1.dcm,1.0,1.0,True,True
15,PATIENT_0592_1.dcm,1.0,0.9,True,True
61,PATIENT_0773_1.dcm,1.0,0.9,True,True
133,PATIENT_0005_1.dcm,1.0,1.0,True,True
111,PATIENT_0297_1.dcm,1.0,1.0,True,True
104,PATIENT_0618_1.dcm,1.0,1.0,True,True
92,PATIENT_0766_1.dcm,1.0,1.0,True,True
98,PATIENT_0713_1.dcm,1.0,1.0,True,True
86,PATIENT_0936_1.dcm,1.0,0.9,True,True
74,PATIENT_0172_1.dcm,1.0,1.0,True,True


In [17]:
def inspect_case(case_filename: str):
    rows = scored_df[scored_df["filename"] == case_filename]
    if rows.empty:
        print(f"Case not found: {case_filename}")
        return
    row = rows.iloc[0]

    print(f"Filename: {row['filename']}")
    print(f"Score: {row['similarity_score_0_to_1']:.2f}/1.0")
    print(f"Strengths: {row['strengths']}")
    print(f"Weaknesses: {row['weaknesses']}")
    print(f"Explanation: {row['explanation']}")
    print("-" * 80)
    print("REFERENCE (radiologist):")
    print(row["reference_description"])
    print("\nGENERATED (rag_pipeline):")
    print(row["generated_description"])

# Example usage:
inspect_case("PATIENT_0002_1.dcm")

Filename: PATIENT_0002_1.dcm
Score: 0.95/1.0
Strengths: - Accurately reports all key findings from the reference (lungs, hila, heart, diaphragm, costophrenic angles).  
- No hallucinated or omitted critical findings.  
- Adds aorta assessment, which is a standard component and not misleading.
Weaknesses: - Omits mention of the lateral projection, which is present in the reference.  
- Slightly different wording ("without distinct infiltrative changes") but does not alter clinical meaning.
Explanation: The AI report closely matches the reference in clinical content and completeness, with only minor differences in phrasing and omission of the lateral projection. The addition of aorta assessment is appropriate and does not detract from accuracy.
--------------------------------------------------------------------------------
REFERENCE (radiologist):
Chest X-ray, PA and lateral projection. Lung fields without infiltrative changes. Hilar shadows on both sides are not enlarged. Heart silhoue

In [18]:
# Optional: export GPT-4 judged comparison table for external analysis.
export_path = ROOT / "data" / "rag_vs_radiologist_comparison_gpt4_judge.csv"
scored_df.to_csv(export_path, index=False)
print(f"Saved comparison table to: {export_path}")

Saved comparison table to: D:\Dev\masters\data\rag_vs_radiologist_comparison_gpt4_judge.csv
